# 04 — Model Evaluation: LegalBERT vs DistilBERT

This notebook evaluates the two fine-tuned models from `03_model_training.ipynb`
on the CUAD test split and compares their clause-extraction performance.

**Metrics**
- **Exact Match (EM)**: % of predictions that match the gold span exactly
  (after lower-casing and punctuation stripping).
- **Token F1**: harmonic mean of token-level precision and recall between
  predicted and gold answer text.

**Evaluation approach**
The same sliding-window tokenisation used during training is applied to the
test set.  Each (clause-type, contract) pair produces one tokenised example;
the model predicts start/end token indices, which are mapped back to character
offsets using the tokeniser's `offset_mapping`.


In [ ]:
import sys
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

sys.path.insert(0, str(Path("../src").resolve()))

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print(f"PyTorch  : {torch.__version__}")
print(f"MPS      : {torch.backends.mps.is_available()}")


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATA_PATH      = Path("../data/processed/cuad_final.csv")
LEGALBERT_OUT  = Path("../models/legalbert-cuad")
DISTILBERT_OUT = Path("../models/distilbert-cuad")

# Evaluation subset: 30 test contracts (manageable inference time)
NUM_TEST_CONTRACTS = 30
MAX_LENGTH = 384
STRIDE     = 128
RANDOM_SEED = 42

for p in [LEGALBERT_OUT, DISTILBERT_OUT]:
    assert p.exists(), f"Model not found: {p}  — run 03_model_training.ipynb first"
print("Both model directories found.")


## Load Test Data

In [ ]:
df      = pd.read_csv(DATA_PATH)
test_df = df[df["split"] == "test"].reset_index(drop=True)

all_test_contracts = test_df["contract_id"].unique()
rng                = np.random.default_rng(RANDOM_SEED)
sampled_ids        = rng.choice(all_test_contracts, size=NUM_TEST_CONTRACTS, replace=False)
test_sample        = test_df[test_df["contract_id"].isin(sampled_ids)].reset_index(drop=True)

print(f"Test contracts available : {len(all_test_contracts)}")
print(f"Contracts sampled        : {len(sampled_ids)}")
print(f"QA pairs in sample       : {len(test_sample):,}")
print(f"  Answerable             : {test_sample['is_answerable'].sum():,}")
print(f"  Unanswerable           : {(~test_sample['is_answerable']).sum():,}")
print(f"Categories covered       : {test_sample['category'].nunique()} / 41")


## Tokenisation Helper

`make_eval_examples` works exactly like `make_qa_examples` from training, but
additionally returns **metadata** for each example — the gold text, context,
offset mapping, and context token range — so we can convert predicted token
indices back into character spans.


In [ ]:
def make_eval_examples(df, tokenizer, max_length=384, stride=128,
                       include_token_type_ids=True):
    """
    Tokenise test rows into sliding-window examples with accompanying metadata.

    Returns:
        examples : list of tokenised dicts (input_ids, attention_mask, ...)
        metadata : list of dicts with gold_text, context, category,
                   contract_id, is_answerable, offsets, ctx_start, ctx_end
    """
    examples = []
    metadata = []
    skipped  = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="tokenising test"):
        encoding = tokenizer(
            row["category"],
            row["context"],
            max_length=max_length,
            truncation="only_second",
            stride=stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
        )

        placed = False
        for chunk_idx in range(len(encoding["input_ids"])):
            seq_ids = encoding.sequence_ids(chunk_idx)
            offsets = encoding["offset_mapping"][chunk_idx]

            ctx_tokens = [i for i, s in enumerate(seq_ids) if s == 1]
            if not ctx_tokens:
                continue
            ctx_start, ctx_end = ctx_tokens[0], ctx_tokens[-1]

            if not row["is_answerable"]:
                if chunk_idx > 0:
                    continue
                ex = {
                    "input_ids":       encoding["input_ids"][chunk_idx],
                    "attention_mask":  encoding["attention_mask"][chunk_idx],
                    "start_positions": 0,
                    "end_positions":   0,
                }
                if include_token_type_ids and "token_type_ids" in encoding.keys():
                    ex["token_type_ids"] = encoding["token_type_ids"][chunk_idx]
                examples.append(ex)
                metadata.append({
                    "gold_text":     str(row["answer_text"]),
                    "context":       row["context"],
                    "category":      row["category"],
                    "contract_id":   row["contract_id"],
                    "is_answerable": bool(row["is_answerable"]),
                    "offsets":       offsets,
                    "ctx_start":     ctx_start,
                    "ctx_end":       ctx_end,
                })
                placed = True
                break

            a_start = int(row["answer_start"])
            a_end   = a_start + len(str(row["answer_text"])) - 1

            if offsets[ctx_start][0] > a_start or offsets[ctx_end][1] < a_end:
                continue

            tok_s = ctx_start
            while tok_s <= ctx_end and offsets[tok_s][0] <= a_start:
                tok_s += 1
            tok_s -= 1

            tok_e = ctx_end
            while tok_e >= ctx_start and offsets[tok_e][1] >= a_end + 1:
                tok_e -= 1
            tok_e += 1

            if tok_s < 0 or tok_e >= max_length or tok_s > tok_e:
                skipped += 1
                continue

            ex = {
                "input_ids":       encoding["input_ids"][chunk_idx],
                "attention_mask":  encoding["attention_mask"][chunk_idx],
                "start_positions": tok_s,
                "end_positions":   tok_e,
            }
            if include_token_type_ids and "token_type_ids" in encoding.keys():
                ex["token_type_ids"] = encoding["token_type_ids"][chunk_idx]
            examples.append(ex)
            metadata.append({
                "gold_text":     str(row["answer_text"]),
                "context":       row["context"],
                "category":      row["category"],
                "contract_id":   row["contract_id"],
                "is_answerable": bool(row["is_answerable"]),
                "offsets":       offsets,
                "ctx_start":     ctx_start,
                "ctx_end":       ctx_end,
            })
            placed = True
            break

        if not placed:
            skipped += 1

    print(f"Eval examples : {len(examples):,}  |  skipped : {skipped}")
    return examples, metadata


## QA Metric Helpers

In [ ]:
import string, unicodedata

def normalise(text):
    text = unicodedata.normalize("NFKC", text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def exact_match(pred, gold):
    return float(normalise(pred) == normalise(gold))

def token_f1(pred, gold):
    pred_toks = normalise(pred).split()
    gold_toks = normalise(gold).split()
    if not pred_toks or not gold_toks:
        return float(pred_toks == gold_toks)
    common = set(pred_toks) & set(gold_toks)
    if not common:
        return 0.0
    prec   = sum(pred_toks.count(t) for t in common) / len(pred_toks)
    recall = sum(gold_toks.count(t) for t in common) / len(gold_toks)
    return 2 * prec * recall / (prec + recall)

def compute_metrics(preds, golds):
    em  = [exact_match(p, g) for p, g in zip(preds, golds)]
    f1s = [token_f1(p, g)    for p, g in zip(preds, golds)]
    return {
        "EM (%)": round(sum(em)  / len(em)  * 100, 2),
        "F1 (%)": round(sum(f1s) / len(f1s) * 100, 2),
        "n":      len(em),
    }

print("Metric helpers defined.")


## Inference Helper

In [ ]:
def run_inference(model, examples, metadata, device, include_token_type_ids=True,
                  batch_size=32):
    """
    Run span-extraction inference on the tokenised examples and decode
    predicted token indices back to answer strings.
    """
    model.eval()
    model = model.to(device)
    predictions = []

    for i in range(0, len(examples), batch_size):
        batch_exs  = examples[i: i + batch_size]
        batch_meta = metadata[i: i + batch_size]

        input_ids      = torch.tensor([e["input_ids"]      for e in batch_exs]).to(device)
        attention_mask = torch.tensor([e["attention_mask"] for e in batch_exs]).to(device)

        inputs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if include_token_type_ids and "token_type_ids" in batch_exs[0]:
            inputs["token_type_ids"] = torch.tensor(
                [e["token_type_ids"] for e in batch_exs]
            ).to(device)

        with torch.no_grad():
            out = model(**inputs)

        for j, meta in enumerate(batch_meta):
            s_idx = out.start_logits[j].argmax().item()
            e_idx = out.end_logits[j].argmax().item()

            offsets   = meta["offsets"]
            ctx_start = meta["ctx_start"]
            ctx_end   = meta["ctx_end"]

            if (s_idx < ctx_start or e_idx < ctx_start or
                    s_idx > ctx_end   or e_idx >= len(offsets) or
                    s_idx > e_idx):
                predictions.append("")
            else:
                char_s = offsets[s_idx][0]
                char_e = offsets[e_idx][1]
                predictions.append(meta["context"][char_s:char_e])

    return predictions

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Inference device: {device}")


## Evaluate LegalBERT

In [ ]:
print("Loading LegalBERT...")
lb_tokenizer = AutoTokenizer.from_pretrained(str(LEGALBERT_OUT))
lb_model     = AutoModelForQuestionAnswering.from_pretrained(str(LEGALBERT_OUT))

print("Tokenising test set for LegalBERT...")
lb_examples, lb_metadata = make_eval_examples(
    test_sample, lb_tokenizer,
    max_length=MAX_LENGTH, stride=STRIDE,
    include_token_type_ids=True,
)


In [ ]:
print("Running LegalBERT inference...")
lb_preds = run_inference(
    lb_model, lb_examples, lb_metadata,
    device=device, include_token_type_ids=True,
)

lb_golds  = [m["gold_text"] for m in lb_metadata]
lb_metrics = compute_metrics(lb_preds, lb_golds)
print(f"\nLegalBERT overall metrics:")
for k, v in lb_metrics.items():
    print(f"  {k}: {v}")


## Evaluate DistilBERT

In [ ]:
print("Loading DistilBERT...")
db_tokenizer = AutoTokenizer.from_pretrained(str(DISTILBERT_OUT))
db_model     = AutoModelForQuestionAnswering.from_pretrained(str(DISTILBERT_OUT))

print("Tokenising test set for DistilBERT...")
db_examples, db_metadata = make_eval_examples(
    test_sample, db_tokenizer,
    max_length=MAX_LENGTH, stride=STRIDE,
    include_token_type_ids=False,   # DistilBERT has no token_type_ids
)


In [ ]:
print("Running DistilBERT inference...")
db_preds = run_inference(
    db_model, db_examples, db_metadata,
    device=device, include_token_type_ids=False,
)

db_golds   = [m["gold_text"] for m in db_metadata]
db_metrics  = compute_metrics(db_preds, db_golds)
print(f"\nDistilBERT overall metrics:")
for k, v in db_metrics.items():
    print(f"  {k}: {v}")


## Results Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model":   ["LegalBERT", "DistilBERT (baseline)"],
    "EM (%)":  [lb_metrics["EM (%)"],  db_metrics["EM (%)"]],
    "F1 (%)":  [lb_metrics["F1 (%)"],  db_metrics["F1 (%)"]],
    "Examples":[lb_metrics["n"],        db_metrics["n"]],
}).set_index("Model")

print("=== Model Comparison ===")
print(comparison.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric in zip(axes, ["EM (%)", "F1 (%)"]):
    bars = ax.bar(comparison.index, comparison[metric],
                  color=["#4C72B0", "#DD8452"], width=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} — LegalBERT vs DistilBERT")
    ax.set_ylim(0, max(comparison[metric].max() * 1.2, 10))
    for bar, val in zip(bars, comparison[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{val:.1f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


## Per-Category Breakdown

In [ ]:
def per_category_metrics(preds, metadata):
    cat_preds = defaultdict(list)
    cat_golds = defaultdict(list)
    for pred, meta in zip(preds, metadata):
        cat_preds[meta["category"]].append(pred)
        cat_golds[meta["category"]].append(meta["gold_text"])
    rows = []
    for cat in sorted(cat_preds):
        m = compute_metrics(cat_preds[cat], cat_golds[cat])
        rows.append({"category": cat, "F1": m["F1 (%)"], "EM": m["EM (%)"], "n": m["n"]})
    return pd.DataFrame(rows).set_index("category")

lb_cat = per_category_metrics(lb_preds, lb_metadata)
db_cat = per_category_metrics(db_preds, db_metadata)

# Side-by-side F1 per category
cat_compare = pd.DataFrame({
    "LegalBERT F1":  lb_cat["F1"],
    "DistilBERT F1": db_cat["F1"],
}).sort_values("LegalBERT F1", ascending=False)

print("Per-category F1 (sorted by LegalBERT):")
print(cat_compare.to_string())


In [ ]:
# Bar chart: per-category F1 (only categories with ≥5 examples to be meaningful)
valid_cats = lb_cat[lb_cat["n"] >= 5].index
plot_data = cat_compare.loc[valid_cats].sort_values("LegalBERT F1")

if len(plot_data) > 0:
    fig, ax = plt.subplots(figsize=(10, max(6, len(plot_data) * 0.35)))
    x = np.arange(len(plot_data))
    w = 0.38
    ax.barh(x + w/2, plot_data["LegalBERT F1"],  w, label="LegalBERT",  color="#4C72B0")
    ax.barh(x - w/2, plot_data["DistilBERT F1"], w, label="DistilBERT", color="#DD8452")
    ax.set_yticks(x)
    ax.set_yticklabels(plot_data.index, fontsize=9)
    ax.set_xlabel("Token F1 (%)")
    ax.set_title("Per-Category F1: LegalBERT vs DistilBERT")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Not enough examples per category for the chart (need >= 5 per category).")


## Error Analysis

In [ ]:
# Show the first 10 cases where LegalBERT was wrong (F1 = 0) on answerable rows
errors = [
    {
        "category":  lb_metadata[i]["category"],
        "gold":      lb_metadata[i]["gold_text"][:120],
        "predicted": lb_preds[i][:120],
        "f1":        round(token_f1(lb_preds[i], lb_metadata[i]["gold_text"]), 3),
    }
    for i in range(len(lb_preds))
    if lb_metadata[i]["is_answerable"]
    and token_f1(lb_preds[i], lb_metadata[i]["gold_text"]) == 0.0
][:10]

if errors:
    print(f"=== LegalBERT: first {len(errors)} zero-F1 answerable examples ===")
    err_df = pd.DataFrame(errors)
    print(err_df.to_string())
else:
    print("No zero-F1 answerable examples found in the test sample.")


## Summary

In [ ]:
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"Test contracts evaluated : {NUM_TEST_CONTRACTS}")
print(f"Total examples           : {len(lb_preds):,}")
print()
print(f"{'Model':<25} {'EM (%)':<10} {'F1 (%)':<10}")
print("-" * 45)
for model_name, m in [("LegalBERT", lb_metrics), ("DistilBERT (baseline)", db_metrics)]:
    print(f"{model_name:<25} {m['EM (%)']:<10} {m['F1 (%)']:<10}")
print()
lb_f1_lead = lb_metrics['F1 (%)'] - db_metrics['F1 (%)']
print(f"LegalBERT F1 advantage over DistilBERT: {lb_f1_lead:+.2f}%")
